# 00 Connection

Connection only: no query against data in this notebook. Question notebooks start with `%run ./00_connection.ipynb`.

- Credentials come from environment variables; only their names appear here.
- The role is read-only (SELECT / USAGE / MONITOR, IMPORTED PRIVILEGES on `SNOWFLAKE` or SELECT on `system` schemas).
- `AS_OF` fixes "now" for every query in the topic. It is read from `ANALYSIS_AS_OF` when set; otherwise it is computed and printed — pin that value in `ANALYSIS_AS_OF` so later notebooks and reruns use the same window.
- Times are UTC: the Snowflake session runs with `TIMEZONE = UTC`, so `AS_OF` compares correctly with `TIMESTAMP_LTZ` columns.

In [ ]:
import datetime as dt
import os
import re

import pandas as pd

PLATFORM = "snowflake"   # or "databricks"
DAYS = 30
# End the window before the slowest view's latency (8 h for Snowflake query attribution, about 1 day for billing).
if os.environ.get("ANALYSIS_AS_OF"):
    AS_OF = dt.datetime.fromisoformat(os.environ["ANALYSIS_AS_OF"])
else:
    AS_OF = (dt.datetime.now(dt.timezone.utc) - dt.timedelta(hours=8)).replace(minute=0, second=0, microsecond=0, tzinfo=None)
    print(f"AS_OF = {AS_OF.isoformat()} (UTC); pin this value in ANALYSIS_AS_OF to reproduce")

In [ ]:
if PLATFORM == "snowflake":
    import snowflake.connector
    snowflake.connector.paramstyle = "qmark"   # server-side binds: values never pasted into SQL text
    conn = snowflake.connector.connect(
        account=os.environ["SNOWFLAKE_ACCOUNT"],
        user=os.environ["SNOWFLAKE_USER"],
        role=os.environ["SNOWFLAKE_ROLE"],
        warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
        authenticator=os.environ.get("SNOWFLAKE_AUTHENTICATOR", "externalbrowser"),
        private_key_file=os.environ.get("SNOWFLAKE_PRIVATE_KEY_FILE"),   # key-pair auth: path only, never the key
        session_parameters={"QUERY_TAG": "analysis-notebook", "TIMEZONE": "UTC"},   # AS_OF is naive UTC
    )
else:
    from databricks import sql as dbsql
    conn = dbsql.connect(
        server_hostname=os.environ["DATABRICKS_SERVER_HOSTNAME"],
        http_path=os.environ["DATABRICKS_HTTP_PATH"],
        access_token=os.environ.get("DATABRICKS_TOKEN"),   # or auth_type="databricks-oauth" for browser sign-in
    )


def q(sql, limit=1000, **params):
    """Run one read-only statement with named :params; returns a DataFrame (at most `limit` rows)."""
    names = [m for m in re.findall(r"(?<![\w:]):(\w+)", sql) if m in params]
    if PLATFORM == "snowflake":
        args = [params[n] for n in names]
        for n in params:
            sql = re.sub(r"(?<![\w:]):" + n + r"\b", "?", sql)
    else:
        args = {n: params[n] for n in set(names)}
    cur = conn.cursor()
    try:
        cur.execute(sql, args)
        rows = cur.fetchmany(limit)
        return pd.DataFrame(rows, columns=[c[0].lower() for c in cur.description])
    finally:
        cur.close()

**Connection test.** Shows the role and compute in use.
- **Grain:** one row = this session
- **Look for:** the read-only role, not an admin or owner role; the expected warehouse.

In [ ]:
if PLATFORM == "snowflake":
    display(q("select current_role() as role_name, current_warehouse() as warehouse, current_version() as version"))
else:
    display(q("select current_user() as user_name, current_catalog() as catalog_name, current_version() as version"))